# Deploying YOLO with FastAPI

Fresh Python 3.11 kernel; complete original five images and original YOLOv3 / YOLOv3-tiny weights, SHA256 verified. Real loopback Uvicorn HTTP, no mocked inference. Separate network instances prevent cvlib’s global-cache model-switch bug. Safe in-memory upload handling and JPEG response encoding replace filename writes. The environment section uses portable venv instead of unavailable conda; web-dependency and JupyterLab installation are executed, interactive browser login is not automated. The notebook kernel supplies inference dependencies installed in setup. Originals preserved below. No API keys.

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,io,time,socket,threading,json
from pathlib import Path
from tempfile import TemporaryDirectory
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check','setuptools==80.9.0','cvlib==0.2.7','opencv-python-headless==4.11.0.86','tensorflow==2.20.0','numpy==2.3.5','pillow==12.0.0','fastapi==0.115.12','uvicorn==0.34.2','python-multipart==0.0.20','httpx==0.28.1','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['TF_CPP_MIN_LOG_LEVEL']='1'
import numpy as np
import cv2
from PIL import Image
from IPython.display import display
from fastapi import FastAPI
import httpx,uvicorn
cv2.setNumThreads(2)
workspace=TemporaryDirectory(prefix='tdb-yolo-fastapi-')
work_dir=Path(workspace.name)
images_dir=work_dir/'images';images_dir.mkdir()
models_dir=work_dir/'models';models_dir.mkdir()
SOURCE_REV='2d82175f1f3eee7c653555f10955331b2436be63'
DARKNET_REV='f6afaabcdf85f77e7aff2ec55c020c0e297c77f9'
def download(url,path,expected=None):
    request=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'})
    with urllib.request.urlopen(request,timeout=180)as response:path.write_bytes(response.read())
    digest=hashlib.sha256(path.read_bytes()).hexdigest()
    if expected is not None:assert digest==expected,(path.name,digest)
    return digest
for name in ['apple.jpg','clock.jpg','oranges.jpg','car.jpg','fruits.jpg']:
    download(f'https://raw.githubusercontent.com/youssefHosni/Practical-Machine-Learning/{SOURCE_REV}/Deploying-Yolo3-Model-on-FastAPI/images/{name}',images_dir/name)
MODEL_HASHES={
 'yolov3.weights':'523e4e69e1d015393a1b0a441cef1d9c7659e3eb2d7e15f793f060a21b32f297',
 'yolov3-tiny.weights':'dccea06f59b781ec1234ddf8d1e94b9519a97f4245748a7d4db75d5b7080a42c',
 'yolov3.cfg':'22489ea38575dfa36c67a90048e8759576416a79d32dc11e15d2217777b9a953',
 'yolov3-tiny.cfg':'84eb7a675ef87c906019ff5a6e0effe275d175adb75100dcb47f0727917dc2c7',
 'coco.names':'634a1132eb33f8091d60f2c346ababe8b905ae08387037aed883953b7329af84'}
for name,digest in MODEL_HASHES.items():
    url=f'https://data.pjreddie.com/files/{name}'if name.endswith('.weights')else f'https://raw.githubusercontent.com/pjreddie/darknet/{DARKNET_REV}/'+('data/'if name=='coco.names'else'cfg/')+name
    download(url,models_dir/name,digest)
def start_local_server(application):
    sock=socket.socket();sock.bind(('127.0.0.1',0))
    port=sock.getsockname()[1]
    server=uvicorn.Server(uvicorn.Config(application,log_level='error',lifespan='off'))
    thread=threading.Thread(target=server.run,kwargs={'sockets':[sock]},daemon=True);thread.start()
    deadline=time.monotonic()+15
    while not server.started:
        if not thread.is_alive()or time.monotonic()>deadline:raise RuntimeError('Local server did not start.')
        time.sleep(.05)
    return server,thread,sock,httpx.Client(base_url=f'http://127.0.0.1:{port}',timeout=60)
def stop_local_server(state):
    server,thread,sock,client=state
    client.close();server.should_exit=True;thread.join(timeout=10);sock.close()
    assert not thread.is_alive()
print('Python:',sys.version.split()[0],'| OpenCV:',cv2.__version__,'| five original source images downloaded.')
print('Both original YOLO weight files, pinned configs and80 labels verified by SHA256.')

In [ ]:
image_files=['apple.jpg','clock.jpg','oranges.jpg','car.jpg']
for image_file in image_files:
    print('Input image:',image_file)
    display(Image.open(images_dir/image_file))

In [ ]:
np.random.seed(42)
from cvlib.object_detection import YOLO
detectors={name:YOLO(str(models_dir/(name+'.weights')),str(models_dir/(name+'.cfg')),str(models_dir/'coco.names'),version=name)for name in ['yolov3-tiny','yolov3']}
locks={name:threading.Lock()for name in detectors}
def detect(image,model='yolov3-tiny',confidence=.5):
    if model not in detectors:raise ValueError('Unsupported model.')
    if not 0<=confidence<=1:raise ValueError('Confidence must be between0 and1.')
    with locks[model]:
        return detectors[model].detect_objects(image,confidence=confidence)
def detect_and_draw_box(filename,model='yolov3-tiny',confidence=.5):
    image=cv2.imread(str(images_dir/filename))
    if image is None:raise ValueError('Image could not be decoded.')
    boxes,labels,scores=detect(image,model,confidence)
    output=image.copy()
    detectors[model].draw_bbox(output,boxes,labels,scores)
    print('Image:',filename,'| model:',model,'| threshold:',confidence)
    print('Detections:',[(label,round(score,6))for label,score in zip(labels,scores)])
    display(Image.fromarray(cv2.cvtColor(output,cv2.COLOR_BGR2RGB)))
    return boxes,labels,scores
print('Separate cached YOLOv3 and YOLOv3-tiny networks and inference locks ready.')

In [ ]:
original_results={}
for image_file in image_files:
    original_results[image_file]=detect_and_draw_box(image_file)
assert 'apple'in original_results['apple.jpg'][1]
assert 'clock'in original_results['clock.jpg'][1]
assert 'orange'in original_results['oranges.jpg'][1]
assert 'car'in original_results['car.jpg'][1]
print('All four source objects detected.')

In [ ]:
fruit_default=detect_and_draw_box('fruits.jpg')

In [ ]:
fruit_lower=detect_and_draw_box('fruits.jpg',confidence=.2)
assert len(fruit_lower[0])>=len(fruit_default[0])
print('Lower threshold detection count:',len(fruit_lower[0]),'| default:',len(fruit_default[0]))

In [ ]:
app=FastAPI()
print('Minimal FastAPI application created.')

In [ ]:
demo_server=start_local_server(app)
demo_client=demo_server[3]
assert demo_client.get('/openapi.json').status_code==200
print('Uvicorn serves the actual application on an isolated loopback port.')

In [ ]:
@app.get('/my-endpoint')
def handle_endpoint():
    return {'message':'GET endpoint is working'}
response=demo_client.get('/my-endpoint')
assert response.status_code==200
print('GET /my-endpoint:',response.json())

In [ ]:
@app.post('/my-other-endpoint')
def handle_other_endpoint(param1:int,param2:str):
    return {'param1':param1,'param2':param2}
response=demo_client.post('/my-other-endpoint',params={'param1':7,'param2':'hello'})
assert response.json()=={'param1':7,'param2':'hello'}
print('POST /my-other-endpoint:',response.json())
stop_local_server(demo_server)
print('Demo Uvicorn server stopped.')

In [ ]:
from enum import Enum
from fastapi import UploadFile,File,HTTPException
from fastapi.responses import StreamingResponse
from PIL import UnidentifiedImageError
print('Upload, validation, enumeration and streaming response imports ready.')

In [ ]:
app=FastAPI(title='Deploying a ML Model with FastAPI')
print('Application:',app.title)

In [ ]:
class Model(str,Enum):
    yolov3tiny='yolov3-tiny'
    yolov3='yolov3'
print('Available models:',[model.value for model in Model])

In [ ]:
@app.get('/')
def home():
    return 'Congratulations! Your API is working as expected.'
print(home())

In [ ]:
MAX_UPLOAD_BYTES=5*1024*1024
MAX_IMAGE_PIXELS=16_000_000
@app.post('/predict')
def prediction(model:Model,file:UploadFile=File(...)):
    filename=file.filename or ''
    if '/'in filename or '\\'in filename:
        raise HTTPException(400,'Provide a filename, not a path.')
    if Path(filename).suffix.lower()not in {'.jpg','.jpeg','.png'}:
        raise HTTPException(415,'Unsupported file provided.')
    payload=file.file.read(MAX_UPLOAD_BYTES+1)
    if len(payload)>MAX_UPLOAD_BYTES:
        raise HTTPException(413,'Image upload exceeds5MiB.')
    try:
        with Image.open(io.BytesIO(payload))as header:
            width,height=header.size
            if width*height>MAX_IMAGE_PIXELS:raise HTTPException(413,'Image dimensions are too large.')
            if header.format not in {'JPEG','PNG'}:raise HTTPException(415,'Only JPEG and PNG data are accepted.')
            header.verify()
    except Image.DecompressionBombError as exc:
        raise HTTPException(413,'Image dimensions are too large.')from exc
    except(UnidentifiedImageError,OSError,ValueError)as exc:
        raise HTTPException(400,'Invalid image data.')from exc
    image=cv2.imdecode(np.frombuffer(payload,dtype=np.uint8),cv2.IMREAD_COLOR)
    if image is None:raise HTTPException(400,'Image could not be decoded.')
    boxes,labels,scores=detect(image,model.value)
    output=image.copy()
    detectors[model.value].draw_bbox(output,boxes,labels,scores)
    ok,encoded=cv2.imencode('.jpg',output)
    if not ok:raise HTTPException(500,'Could not encode prediction.')
    return StreamingResponse(io.BytesIO(encoded.tobytes()),media_type='image/jpeg')
api_server=start_local_server(app)
client=api_server[3]
assert client.get('/').json()==home()
http_results={}
for selected_model in Model:
    response=client.post('/predict',params={'model':selected_model.value},files={'file':('apple.jpg',(images_dir/'apple.jpg').read_bytes(),'image/jpeg')})
    assert response.status_code==200 and response.headers['content-type']=='image/jpeg'
    annotated=cv2.imdecode(np.frombuffer(response.content,np.uint8),cv2.IMREAD_COLOR)
    assert annotated is not None
    http_results[selected_model.value]=response.content
    display(Image.fromarray(cv2.cvtColor(annotated,cv2.COLOR_BGR2RGB)))
    print('POST /predict | model:',selected_model.value,'| status:',response.status_code,'| media:',response.headers['content-type'],'| output shape:',annotated.shape)

In [ ]:
clone_dir=work_dir/'Practical-Machine-Learning'
subprocess.run(['git','clone','--quiet','https://github.com/youssefHosni/Practical-Machine-Learning.git',str(clone_dir)],check=True)
subprocess.run(['git','-C',str(clone_dir),'checkout','--quiet','--detach',SOURCE_REV],check=True)
commit=subprocess.check_output(['git','-C',str(clone_dir),'rev-parse','HEAD'],text=True).strip()
assert commit==SOURCE_REV
print('Source repository cloned at:',commit)

In [ ]:
import venv
deployment_env=work_dir/'deployment-tutorial'
venv.EnvBuilder(with_pip=True,symlinks=os.name!='nt').create(deployment_env)
deployment_python=deployment_env/('Scripts/python.exe'if os.name=='nt'else'bin/python')
print('Created isolated deployment-tutorial environment.')

In [ ]:
version=subprocess.check_output([str(deployment_python),'--version'],text=True).strip()
assert version.startswith('Python3.11'.replace('Python3','Python 3'))
print('Selected environment interpreter:',version)

In [ ]:
requirements=work_dir/'requirements.txt'
requirements.write_text('fastapi==0.115.12\nuvicorn==0.34.2\npython-multipart==0.0.20\n',encoding='utf-8')
subprocess.run([str(deployment_python),'-m','pip','install','--quiet','--disable-pip-version-check','-r',str(requirements)],check=True)
smoke=subprocess.check_output([str(deployment_python),'-c','import fastapi,uvicorn; import python_multipart; print(fastapi.__version__,uvicorn.__version__)'],text=True).strip()
assert smoke=='0.115.12 0.34.2'
print('Pinned web deployment dependency smoke test:',smoke)

In [ ]:
subprocess.run([str(deployment_python),'-m','pip','install','--quiet','--disable-pip-version-check','jupyterlab==4.4.3'],check=True)
jupyter_version=subprocess.check_output([str(deployment_python),'-m','jupyterlab','--version'],text=True).strip()
assert jupyter_version=='4.4.3'
print('JupyterLab installed:',jupyter_version)
print('Launch interactively with the environment interpreter: python -m jupyterlab')

In [ ]:
assert len(detectors['yolov3'].net.getLayerNames())>len(detectors['yolov3-tiny'].net.getLayerNames())
assert len(detectors['yolov3'].labels)==80
# The independent library entry point must match each correctly selected cached model.
import cvlib.object_detection as od
od.dest_dir=str(models_dir)
(models_dir/'yolov3_classes.txt').write_text((models_dir/'coco.names').read_text())
for model in Model:
    image=cv2.imread(str(images_dir/'apple.jpg'))
    od.initialize=True
    reference=od.detect_common_objects(image,confidence=.5,model=model.value)
    actual=detect(image,model.value)
    assert reference[0:2]==actual[0:2]
    np.testing.assert_allclose(reference[2],actual[2],rtol=1e-6)
    annotated=image.copy()
    detectors[model.value].draw_bbox(annotated,*actual)
    ok,jpeg=cv2.imencode('.jpg',annotated)
    assert ok and jpeg.tobytes()==http_results[model.value]
tests=[('bad.txt',b'bad','yolov3-tiny',415),('bad.jpg',b'bad','yolov3-tiny',400),('../escape.jpg',b'bad','yolov3-tiny',400),('bad.jpg',b'x'*(MAX_UPLOAD_BYTES+1),'yolov3-tiny',413),('apple.jpg',(images_dir/'apple.jpg').read_bytes(),'unknown',422)]
for filename,payload,model,status in tests:
    response=client.post('/predict',params={'model':model},files={'file':(filename,payload,'application/octet-stream')})
    assert response.status_code==status,(filename,response.status_code,response.text)
png=io.BytesIO();Image.open(images_dir/'apple.jpg').save(png,format='PNG')
response=client.post('/predict',params={'model':'yolov3-tiny'},files={'file':('APPLE.PNG',png.getvalue(),'image/png')})
assert response.status_code==200 and response.headers['content-type']=='image/jpeg'
assert Image.open(io.BytesIO(response.content)).format=='JPEG'
assert client.post('/predict',params={'model':'yolov3-tiny'}).status_code==422
assert client.post('/predict',params={'model':'yolov3-tiny'},files={'file':('folder\\bad.jpg',b'bad','image/jpeg')}).status_code==400
# Oversized PNG dimensions are checked from the header before allocating pixels.
import struct,zlib
def png_chunk(kind,data):
    return struct.pack('>I',len(data))+kind+data+struct.pack('>I',zlib.crc32(kind+data)&0xffffffff)
for dimensions in [(5000,4000),(20000,20000)]:
    payload=b'\x89PNG\r\n\x1a\n'+png_chunk(b'IHDR',struct.pack('>IIBBBBB',*dimensions,8,2,0,0,0))+png_chunk(b'IEND',b'')
    response=client.post('/predict',params={'model':'yolov3-tiny'},files={'file':('huge.png',payload,'image/png')})
    assert response.status_code==413,(dimensions,response.status_code)
gif=io.BytesIO();Image.new('RGB',(2,2)).save(gif,format='GIF')
assert client.post('/predict',params={'model':'yolov3-tiny'},files={'file':('disguised.jpg',gif.getvalue(),'image/jpeg')}).status_code==415
from concurrent.futures import ThreadPoolExecutor
def concurrent_request(index):
    model=['yolov3-tiny','yolov3'][index%2]
    result=client.post('/predict',params={'model':model},files={'file':('same-name.jpg',(images_dir/'apple.jpg').read_bytes(),'image/jpeg')})
    assert result.status_code==200 and result.content==http_results[model]
with ThreadPoolExecutor(max_workers=2)as pool:list(pool.map(concurrent_request,range(4)))
schema=client.get('/openapi.json').json()
assert '/predict'in schema['paths']
print('Both model networks match the independent cvlib entry point and byte-exact HTTP JPEG output.')
print('Real Uvicorn GET/POST, both models, invalid model, missing upload, both path separators, extension, corrupt bytes, byte/pixel limits, disguised GIF, concurrent same-name uploads and PNG→JPEG checks passed.')
stop_local_server(api_server)
workspace.cleanup()
print('Owned Uvicorn server and temporary workspace cleaned up.')

## Original source code

### Original block 1

```text
# Some example images
image_files = [
    'apple.jpg',
    'clock.jpg',
    'oranges.jpg',
    'car.jpg'
]

for image_file in image_files:
    print(f"\nDisplaying image: {image_file}")
    display(Image(filename=f"images/{image_file}"))
```

### Original block 2

```text
import cv2
import cvlib as cv
from cvlib.object_detection import draw_bbox

def detect_and_draw_box(filename, model="yolov3-tiny", confidence=0.5):
    """Detects common objects on an image and creates a new image with bounding boxes.

    Args:
        filename (str): Filename of the image.
        model (str): Either "yolov3" or "yolov3-tiny". Defaults to "yolov3-tiny".
        confidence (float, optional): Desired confidence level. Defaults to 0.5.
    """
    
    # Images are stored under the images/ directory
    img_filepath = f'images/{filename}'
    
    # Read the image into a numpy array
    img = cv2.imread(img_filepath)
    
    # Perform the object detection
    bbox, label, conf = cv.detect_common_objects(img, confidence=confidence, model=model)
    
    # Print current image's filename
    print(f"========================\nImage processed: {filename}\n")
    
    # Print detected objects with confidence level
    for l, c in zip(label, conf):
        print(f"Detected object: {l} with confidence level of {c}\n")
    
    # Create a new image that includes the bounding boxes
    output_image = draw_bbox(img, bbox, label, conf)
    
    # Save the image in the directory images_with_boxes
    cv2.imwrite(f'images_with_boxes/{filename}', output_image)
    
    # Display the image with bounding boxes
    display(Image(f'images_with_boxes/{filename}'))
```

### Original block 3

```text
for image_file in image_files:
    detect_and_draw_box(image_file)
```

### Original block 4

```text
detect_and_draw_box("fruits.jpg")
```

### Original block 5

```text
detect_and_draw_box("fruits.jpg", confidence=0.2)
```

### Original block 6

```text
app = FastAPI()
```

### Original block 7

```text
uvicorn.run(app)
```

### Original block 8

```text
@app.get("/my-endpoint")
def handle_endpoint():
    ...
    ...
```

### Original block 9

```text
@app.post("/my-other-endpoint")
def handle_other_endpoint(param1: int, param2: str):
    ...
    ...
```

### Original block 10

```text
import io
import uvicorn
import numpy as np
import nest_asyncio
from enum import Enum
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse
```

### Original block 11

```text
# Assign an instance of the FastAPI class to the variable "app".
# You will interact with your api using this instance.
app = FastAPI(title='Deploying a ML Model with FastAPI')
```

### Original block 12

```text
# List available models using Enum for convenience. 
# This is useful when the options are pre-defined.
class Model(str, Enum):
    yolov3tiny = "yolov3-tiny"
    yolov3 = "yolov3"
```

### Original block 13

```text
# By using @app.get("/") you are allowing the GET method to work for the / endpoint.
@app.get("/")
def home():
    return "Congratulations! Your API is working as expected."
```

### Original block 14

```text
@app.post("/predict") 
def prediction(model: Model, file: UploadFile = File(...)):

    # 1. VALIDATE INPUT FILE
    filename = file.filename
    fileExtension = filename.split(".")[-1] in ("jpg", "jpeg", "png")
    if not fileExtension:
        raise HTTPException(status_code=415, detail="Unsupported file provided.")
    
    # 2. TRANSFORM RAW IMAGE INTO CV2 image
    
    # Read image as a stream of bytes
    image_stream = io.BytesIO(file.file.read())
    
    # Start the stream from the beginning (position zero)
    image_stream.seek(0)
    
    # Write the stream of bytes into a numpy array
    file_bytes = np.asarray(bytearray(image_stream.read()), dtype=np.uint8)
    
    # Decode the numpy array as an image
    image = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    
    
    # 3. RUN OBJECT DETECTION MODEL
    
    # Run object detection
    bbox, label, conf = cv.detect_common_objects(image, model=model)
    
    # Create image that includes bounding boxes and labels
    output_image = draw_bbox(image, bbox, label, conf)
    
    # Save it in a folder within the server
    cv2.imwrite(f'images_uploaded/{filename}', output_image)
    
    
    # 4. STREAM THE RESPONSE BACK TO THE CLIENT
    
    # Open the saved image for reading in binary mode
    file_image = open(f'images_uploaded/{filename}', mode="rb")
    
    # Return the image as a stream specifying media type
    return StreamingResponse(file_image, media_type="image/jpeg")
```

### Original block 15

```text
git clone https://github.com/youssefHosni/Practical-Machine-Learning.git
```

### Original block 16

```text
conda create --name deployment-tutorial python=3.8
```

### Original block 17

```text
conda activate deployment-tutorial
```

### Original block 18

```text
pip install -r requirements.txt
```

### Original block 19

```text
jupyter lab
```